## Load NASA Data

In [8]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import os

# Plot settings
plt.style.use("seaborn-v0_8")


def fetch_nasa_daily_ghi(lat, lon, start, end):
    """
    Fetch daily GHI from NASA POWER for given lat/lon and date range.
    start, end: YYYYMMDD strings
    """
    base_url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {
        "start": start,
        "end": end,
        "latitude": lat,
        "longitude": lon,
        "parameters": "ALLSKY_SFC_SW_DWN,T2M",
        "community": "RE",
        "format": "JSON"
    }
    r = requests.get(base_url, params=params)
    r.raise_for_status()
    data = r.json()

    records = data["properties"]["parameter"]["ALLSKY_SFC_SW_DWN"]
    temps = data["properties"]["parameter"]["T2M"]

    df = pd.DataFrame({
        "date": list(records.keys()),
        "ghi_kwh_m2": list(records.values()),
    })
    df["temp_c"] = df["date"].map(temps)
    df["date"] = pd.to_datetime(df["date"])
    df["lat"] = lat
    df["lon"] = lon
    return df.sort_values("date").reset_index(drop=True)

os.makedirs("kaggle/working/data/raw", exist_ok=True)
os.makedirs("kaggle/working/models", exist_ok=True)


locations = {
    "Nairobi": (-1.286389, 36.817223),
    "Lagos": (6.524379, 3.379206),
    "Johannesburg": (-26.204103, 28.047304)
}

start = "20240101"
end = "20241231"

all_dfs = []
for name, (lat, lon) in locations.items():
    print("Fetching", name)
    df_loc = fetch_nasa_daily_ghi(lat, lon, start, end)
    df_loc["city"] = name
    all_dfs.append(df_loc)
    df_loc.to_csv(f"kaggle/working/data/raw/nasa_{name.lower()}_2024.csv", index=False)

df_all = pd.concat(all_dfs, ignore_index=True)
df_all.to_csv("kaggle/working/data/raw/nasa_multi_cities_2024.csv", index=False)
df_all.head()

Fetching Nairobi
Fetching Lagos
Fetching Johannesburg


,date,ghi_kwh_m2,temp_c,lat,lon,city
0,2024-01-01,5.3304,20.89,-1.286389,36.817223,Nairobi
1,2024-01-02,4.8876,20.32,-1.286389,36.817223,Nairobi
2,2024-01-03,5.3273,20.45,-1.286389,36.817223,Nairobi
3,2024-01-04,5.5294,20.66,-1.286389,36.817223,Nairobi
4,2024-01-05,4.7942,20.84,-1.286389,36.817223,Nairobi


In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb
import joblib

df = pd.read_csv("kaggle/working/data/raw/nasa_multi_cities_2024.csv", parse_dates=["date"])
df.head()

,date,ghi_kwh_m2,temp_c,lat,lon,city
0,2024-01-01,5.3304,20.89,-1.286389,36.817223,Nairobi
1,2024-01-02,4.8876,20.32,-1.286389,36.817223,Nairobi
2,2024-01-03,5.3273,20.45,-1.286389,36.817223,Nairobi
3,2024-01-04,5.5294,20.66,-1.286389,36.817223,Nairobi
4,2024-01-05,4.7942,20.84,-1.286389,36.817223,Nairobi


## Feature engineering

In [10]:
# Calendar features
df["day_of_year"] = df["date"].dt.dayofyear
df["sin_doy"] = np.sin(2 * np.pi * df["day_of_year"] / 365.0)
df["cos_doy"] = np.cos(2 * np.pi * df["day_of_year"] / 365.0)

# Simulate per‑kW energy target (1 kW system, PR = 0.75)
PR = 0.75
SYSTEM_SIZE_KW = 1.0
df["target_kwh_per_kw"] = df["ghi_kwh_m2"] * SYSTEM_SIZE_KW * PR

df[["ghi_kwh_m2", "temp_c", "target_kwh_per_kw"]].head()

,ghi_kwh_m2,temp_c,target_kwh_per_kw
0,5.3304,20.89,3.997800
1,4.8876,20.32,3.665700
2,5.3273,20.45,3.995475
3,5.5294,20.66,4.147050
4,4.7942,20.84,3.595650


##  Train/test split + XGBoost

In [11]:
feature_cols = ["ghi_kwh_m2", "temp_c", "sin_doy", "cos_doy"]

X = df[feature_cols]
y = df["target_kwh_per_kw"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=42
)

model_xgb = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model_xgb.fit(X_train, y_train)

y_pred = model_xgb.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae, rmse

(0.05305047522284767, np.float64(0.07430242944896967))

## Save model + feature list

In [12]:
joblib.dump(model_xgb, "kaggle/working/models/xgb_baseline.pkl")
pd.Series(feature_cols).to_csv("kaggle/working/models/feature_columns.csv", index=False)